# 航班清洗与同机衔接

本 notebook 说明五个月 BTS 原始航班如何转成可审计的清洗表与连续航段。完整处理逻辑只维护在 `scripts/prepare_flights.py`；notebook 负责讲解并调用同一份代码。

This notebook explains the five-month BTS cleaning and same-aircraft linking workflow. The script is the single source of processing logic.


In [1]:
ROOT = __import__('pathlib').Path.cwd()
RUN_DIR = ROOT / 'experiments/runs/R0015'

## 1. 输入与目录

输入为五个月 BTS ZIP 与机场参考表。原始 BTS AirportID 是稳定机场实体；脚本用本期唯一代码到 ID 的映射核对，PBI/DJT 可映射同一 AirportID。代码日期范围核验另外保存在准备审计运行中，不把静态代码映射说成逐日期匹配。


## 2. 规则与边界

- 保留取消、备降、缺尾号及缺失/异常时刻记录，先依尾号和计划出发 UTC 排序建立相邻序列；它们不会被跳过来制造虚假的前后航段。
- 使用出发机场当地日期和 HHMM 构造计划 UTC；`2400` 表示次日 00:00。DST 歧义/不存在的本地时间视为无效。
- 按原逻辑核对计划到达钟表、实际出发/到达钟表、实际历时及 BTS 延误字段，容差为 1 分钟。
- 负的 `DepDelay`/`ArrDelay` 原值保留；`DepDelayMinutes`/`ArrDelayMinutes` 是非负分钟口径。衔接变化 `delta_delay_minutes` = 下游非负出发延误分钟 − 上游非负到达延误分钟。
- 有效连接要求同一稳定机场 ID 连续、双段正常且时间有效、顺序无冲突、计划周转为 0–1440 分钟、实际地面时间非负。跨月相邻航班照常保留。预测时点是上游实际到达。

Cancellations, diversions, missing tails and invalid times remain present while the sequence is constructed. Delay values and nonnegative delay minutes are both retained. The prediction timestamp is the upstream actual arrival.


## 3. 运行与输出

正式准备通过统一入口执行，并记录独立 run_id。此 Notebook 读取已完成准备的汇总与排除原因，避免重复全量写文件。若尚未准备，先运行 `python scripts/run_pipeline.py --config configs/prepare.json --run-id 新编号`。

原始文件保持不变；准备输出包含逐行原始定位、候选排除原因和有效衔接。


In [2]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
package_root = next(p for p in [cwd, *cwd.parents] if (p / 'scripts' / 'prepare_flights.py').is_file())
sys.path.insert(0, str(package_root / 'scripts'))
from prepare_flights import self_check
self_check()
evidence_dir = Path(globals().get('RUN_DIR', package_root / 'tables'))
summary_path = evidence_dir / 'cleaning_summary.json'
assert summary_path.is_file(), '先通过统一入口运行全量数据准备。'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
display(pd.Series(summary['summary'], name='value').to_frame())
display(pd.read_csv(evidence_dir / 'connection_exclusions.csv'))


,value
raw_flight_rows,3.057590e+06
duplicate_key_rows,0.000000e+00
missing_tail_rows,5.171000e+03
cancelled_rows,4.891900e+04
diverted_rows,1.167900e+04
normal_operation_rows,2.996992e+06
normal_time_valid_rows,2.996940e+06
airport_count,3.550000e+02
candidate_connections,3.046298e+06
accepted_connections,2.821852e+06


,reason,pairs,share_of_candidates
0,accepted,2821852,0.926322
1,qa_cohort_in_scope,57302,0.018810
2,normal_both,38318,0.012579
3,scheduled_gap_nonnegative,36673,0.012039
4,scheduled_gap_in_scope,34379,0.011286
5,airport_continuity,32976,0.010825
6,actual_order_consistent,23277,0.007641
7,unique_schedule,1419,0.000466
8,time_valid_both,93,0.000031
9,actual_ground_nonnegative,9,0.000003


## 4. 审查边界

表格为实际准备的结构性计数，不展示 8 月目标分布或模型误差。RQ2 还须严格要求上游到达早于下游出发；基础连接允许相同时刻，不能直接当成最终预测样本。全期事后质量规则是否影响开发样本，另见 prefix QA 审计及后续数据版本决定。
